# Robotics Autoresearch: Progress Analysis

Tracking MPC controller optimization for Unitree H1 forward speed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load results
df = pd.read_csv("results.tsv", sep="\t")
print(f"Total experiments: {len(df)}")
print(f"  keep:    {(df['status'] == 'keep').sum()}")
print(f"  discard: {(df['status'] == 'discard').sum()}")
print(f"  crash:   {(df['status'] == 'crash').sum()}")
print(f"\nBest speed: {df.loc[df['status'] == 'keep', 'avg_speed_mps'].max():.4f} m/s")
print(f"Total wall time: {df['wall_time_s'].sum() / 3600:.1f} hours")
df.tail(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Plot all experiments
colors = {"keep": "green", "discard": "gray", "crash": "red"}
for status, group in df.groupby("status"):
    ax.scatter(group.index, group["avg_speed_mps"],
               c=colors.get(status, "blue"), label=status,
               s=40, alpha=0.7, zorder=3)

# Running maximum line (keep experiments only)
keep_mask = df["status"] == "keep"
if keep_mask.any():
    keep_speeds = df.loc[keep_mask, "avg_speed_mps"]
    running_max = keep_speeds.expanding().max()
    ax.step(keep_speeds.index, running_max, where="post",
            color="green", linewidth=2, linestyle="--",
            label="running max", zorder=2)

ax.set_xlabel("Experiment #")
ax.set_ylabel("Average Forward Speed (m/s)")
ax.set_title("H1 MPC Optimization Progress")
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="black", linewidth=0.5, linestyle="-")
plt.tight_layout()
plt.show()

In [ ]:
# Wall time per experiment
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df.index, df["wall_time_s"], color="steelblue", alpha=0.7)
ax.set_xlabel("Experiment #")
ax.set_ylabel("Wall Time (s)")
ax.set_title("Experiment Duration")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary
print(f"Mean wall time: {df['wall_time_s'].mean():.1f}s")
print(f"Median wall time: {df['wall_time_s'].median():.1f}s")